# FlashEats — Class 7 Challenge
## Model the Business Workflow with Data

### Client question
> **“Show us where in the workflow delay accumulates, how customers react, what interventions we make, and which metrics we should use to improve the project KPI.”**

Do not repeat source discovery, retrieval, or data-cleaning work. Today the goal is to build a useful business-workflow model.

In [1]:
import os, json, sqlite3
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 200)


def find_pack_root():
    """Pack root = folder with database/flasheats.db AND the Class 6/7 files (the updated instructor pack).

    Search order: $FLASHEATS_PACK, Challenges/flasheats-classroom-pack (created by get_pack.sh),
    the working directory and its parents, ~/flasheats-classroom-pack, then the Colab location.
    """
    candidates = [Path(os.environ["FLASHEATS_PACK"])] if os.environ.get("FLASHEATS_PACK") else []
    candidates += [Path.cwd() / "flasheats-classroom-pack", Path.cwd() / "Challenges" / "flasheats-classroom-pack",
                   Path.cwd(), *Path.cwd().parents, Path.home() / "flasheats-classroom-pack",
                   Path("/content/FlashEats_Classroom_Pack")]
    for folder in candidates:
        if (folder / "database" / "flasheats.db").exists() and (folder / "data" / "order_outcomes.csv").exists():
            return folder
    raise FileNotFoundError("FlashEats pack not found. Run `bash Challenges/get_pack.sh` or set FLASHEATS_PACK.")


BASE = find_pack_root()
OUTPUT = Path.cwd() / "output" if (Path.cwd() / "get_pack.sh").exists() else Path.cwd() / "Challenges" / "output"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Pack:", os.path.relpath(BASE))

import numpy as np

Pack: flasheats-classroom-pack


In [2]:
con=sqlite3.connect(BASE/"database"/"flasheats.db")
orders=pd.read_sql("SELECT * FROM orders",con)
customers=pd.read_sql("SELECT * FROM customers",con)
restaurants=pd.read_sql("SELECT * FROM restaurants",con)
drivers=pd.read_sql("SELECT * FROM drivers",con)
tickets=pd.read_csv(BASE/"data"/"support_tickets.csv")
customer_actions=pd.read_csv(BASE/"data"/"customer_app_actions.csv")
interventions=pd.read_csv(BASE/"data"/"order_interventions.csv")
outcomes=pd.read_csv(BASE/"data"/"order_outcomes.csv")
with open(BASE/"data"/"class7_model_brief.json") as f: model_brief=json.load(f)
print("orders",orders.shape,"actions",customer_actions.shape,"interventions",interventions.shape,"outcomes",outcomes.shape)
print("Project KPI:",model_brief["project_kpi"])

orders (1603, 13) actions (2365, 6) interventions (430, 6) outcomes (1600, 6)
Project KPI: Reduce late delivery rate


# Challenge 1 — Reconstruct the order lifecycle

Choose 3 orders:
- one delivered on time,
- one delivered late,
- one with an intervention.

Build a timeline with:

`event_time | event_type | actor | source_system`

Include as many lifecycle events as the data supports.

### Hint
Start from one `order_id`, collect events from each source, then sort by time.

In [3]:
base_orders=orders.drop_duplicates("order_id",keep="first")
late_order=outcomes[outcomes.late_flag==1].order_id.iloc[0]
on_time_order=outcomes[outcomes.late_flag==0].order_id.iloc[0]
intervention_order=interventions.order_id.iloc[0]
print("late",late_order,"on-time",on_time_order,"intervention",intervention_order)

# TODO: build_order_timeline(order_id)

late O00001 on-time O00003 intervention O00781


In [4]:
events = pd.read_csv(BASE / "data" / "order_events.csv")
interactions = pd.read_csv(BASE / "data" / "customer_interactions.csv")
dispatch = pd.DataFrame(json.loads((BASE / "api" / "dispatch_data.json").read_text()))
tickets_unique = tickets.drop_duplicates()          # T00013 is stored twice (exact copy)


def build_order_timeline(order_id):
    """Collect every lifecycle event for one order from each source system, sorted by time."""
    parts = [
        events.loc[events.order_id == order_id].assign(event_type=lambda d: d.event_type, actor=lambda d: d.actor_type + ":" + d.actor_id.fillna("?"))
              [["event_time", "event_type", "actor", "source_system"]],
        customer_actions.loc[customer_actions.order_id == order_id].assign(
            event_time=lambda d: d.action_at, event_type=lambda d: "APP_" + d.action_type,
            actor=lambda d: "customer:" + d.customer_id, source_system="customer_app")[["event_time", "event_type", "actor", "source_system"]],
        tickets_unique.loc[tickets_unique.order_id == order_id].assign(
            event_time=lambda d: d.created_at, event_type=lambda d: "TICKET_" + d.category.str.strip(),
            actor="customer", source_system="support")[["event_time", "event_type", "actor", "source_system"]],
        interventions.loc[interventions.order_id == order_id].assign(
            event_time=lambda d: d.intervention_at, event_type=lambda d: "INTERVENTION_" + d.intervention_type + " (" + d.reason + ")",
            actor=lambda d: d.initiated_by, source_system="interventions")[["event_time", "event_type", "actor", "source_system"]],
    ]
    dsp = dispatch.loc[dispatch.order_id == order_id]
    for col, label in [("assigned_at", "DISPATCH_ASSIGNED"), ("reassigned_at", "DISPATCH_REASSIGNED"),
                       ("estimated_pickup_at", "DISPATCH_EST_PICKUP (planned)"), ("current_delivery_eta", "DISPATCH_CURRENT_ETA (planned)")]:
        sub = dsp[dsp[col].notna()]
        parts.append(sub.assign(event_time=sub[col], event_type=label,
                     actor=lambda d: "driver:" + d.driver_id, source_system="dispatch")[["event_time", "event_type", "actor", "source_system"]])
    o = orders.drop_duplicates("order_id").set_index("order_id").loc[order_id]
    parts.append(pd.DataFrame([{"event_time": o.promised_eta, "event_type": "PROMISED_ETA (planned)", "actor": "order_platform", "source_system": "orders"}]))
    tl = pd.concat(parts, ignore_index=True)
    tl["event_time"] = pd.to_datetime(tl["event_time"], format="mixed")
    return tl.sort_values("event_time").reset_index(drop=True)


for label, oid in [("ON TIME", on_time_order), ("LATE", late_order), ("WITH INTERVENTION", intervention_order)]:
    out = outcomes.set_index("order_id").loc[oid]
    print(f"\n=== {label}: {oid} | outcome={out.outcome_bucket} delay_min={out.delay_min}")
    display(build_order_timeline(oid))


=== ON TIME: O00003 | outcome=delivered_on_time delay_min=-4.41


,event_time,event_type,actor,source_system
0,2026-08-01 19:35:00.000000,ORDER_CREATED,customer:C0855,order_platform
1,2026-08-01 19:35:00.002443,DISPATCH_ASSIGNED,driver:D039,dispatch
2,2026-08-01 19:53:00.000000,DISPATCH_EST_PICKUP (planned),driver:D039,dispatch
3,2026-08-01 19:57:50.820366,PICKED_UP,driver:D039,order_platform
4,2026-08-01 20:13:28.061191,DELIVERED,driver:D039,order_platform
5,2026-08-01 20:17:52.941967,PROMISED_ETA (planned),order_platform,orders
6,2026-08-01 20:22:59.204700,DISPATCH_CURRENT_ETA (planned),driver:D039,dispatch



=== LATE: O00001 | outcome=delivered_late delay_min=10.82


,event_time,event_type,actor,source_system
0,2026-08-13 12:56:00.000000,ORDER_CREATED,customer:C0168,order_platform
1,2026-08-13 13:01:09.659644,DISPATCH_ASSIGNED,driver:D103,dispatch
2,2026-08-13 13:08:00.000000,INTERVENTION_DRIVER_REASSIGNMENT (driver_unavailable),dispatch,interventions
3,2026-08-13 13:14:00.000000,DISPATCH_EST_PICKUP (planned),driver:D103,dispatch
4,2026-08-13 13:21:00.000000,APP_ETA_VIEWED,customer:C0168,customer_app
5,2026-08-13 13:35:49.825561,PICKED_UP,driver:D103,order_platform
6,2026-08-13 14:11:36.000000,PROMISED_ETA (planned),order_platform,orders
7,2026-08-13 14:19:15.546424,DISPATCH_CURRENT_ETA (planned),driver:D103,dispatch
8,2026-08-13 14:20:36.000000,APP_SUPPORT_OPENED,customer:C0168,customer_app
9,2026-08-13 14:22:25.035899,DELIVERED,driver:D103,order_platform



=== WITH INTERVENTION: O00781 | outcome=delivered_late delay_min=13.07


,event_time,event_type,actor,source_system
0,2026-08-22 12:33:00.000000,ORDER_CREATED,customer:C0103,order_platform
1,2026-08-22 12:40:42.931820,DISPATCH_ASSIGNED,driver:D103,dispatch
2,2026-08-22 12:51:00.000000,DISPATCH_EST_PICKUP (planned),driver:D103,dispatch
3,2026-08-22 12:54:00.000000,APP_ETA_VIEWED,customer:C0103,customer_app
4,2026-08-22 12:57:00.000000,INTERVENTION_PRIORITY_DISPATCH (late_risk),operations,interventions
5,2026-08-22 13:11:17.309762,PICKED_UP,driver:D103,order_platform
6,2026-08-22 13:48:36.000000,PROMISED_ETA (planned),order_platform,orders
7,2026-08-22 13:54:02.886389,DISPATCH_CURRENT_ETA (planned),driver:D103,dispatch
8,2026-08-22 14:01:40.289785,DELIVERED,driver:D103,order_platform


**What the three timelines show:**
- **O00003 (on time):** created → assigned → picked up 5 min after dispatch's planned pickup → delivered **4.4 minutes before** the promise. No customer app activity at all.
- **O00001 (late):**
  - Picked up at 13:35, **22 min after dispatch's planned pickup** (13:14).
  - Dispatch's customer-facing ETA (14:19) had already slipped past the promise (14:11).
  - The customer checked the ETA at 13:21, opened support at 14:20, and the order arrived **10.8 min late**.
  - **Source conflict:** the intervention log records a `DRIVER_REASSIGNMENT` (driver_unavailable) at 13:08, but dispatch shows no reassignment and the same driver (D103) throughout. One of the two systems is wrong, which is exactly why the model needs a named source of truth.
- **O00781 (intervention):**
  - Operations issued `PRIORITY_DISPATCH` for `late_risk` at 12:57, *before* pickup, so it was preventive in timing.
  - Pickup still came 20 min after plan, and the order was **13.1 min late**.
  - The timeline shows *when* each intervention happened; Challenge 5 shows why that matters (an action after delivery cannot have prevented lateness).
- **Planned vs observed:** events marked *(planned)* are promises or estimates, not facts. Mixing them with observed events on one timeline is only safe because they are labelled.

# Challenge 2 — Define the canonical project model

Your model must support:
1. customer → orders
2. order → customer interactions
3. order → support interactions
4. order → interventions
5. order → outcome

For each table, document:
- primary key,
- important foreign keys,
- grain.

Then explain why this model is better for the project than mirroring every source-system table.

In [5]:
sources={"orders":orders,"customer_actions":customer_actions,"support_tickets":tickets,"interventions":interventions,"outcomes":outcomes}
for name,df in sources.items():
    print(name,df.shape); display(df.head(2))
# TODO: document grain + relationships

orders (1603, 13)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear


customer_actions (2365, 6)


,action_id,order_id,customer_id,action_type,action_at,channel
0,ACT-00001,O01044,C0249,ETA_VIEWED,2026-08-05T22:11:00,mobile_app
1,ACT-00002,O01334,C0692,ETA_VIEWED,2026-08-01T17:48:00,mobile_app


support_tickets (202, 5)


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.


interventions (430, 6)


,intervention_id,order_id,intervention_type,intervention_at,initiated_by,reason
0,INT-00001,O00781,PRIORITY_DISPATCH,2026-08-22T12:57:00,operations,late_risk
1,INT-00002,O01476,CUSTOMER_CREDIT,2026-08-01T23:46:36.200640,support,support_resolution


outcomes (1600, 6)


,order_id,final_status_norm,delivered_flag,late_flag,delay_min,outcome_bucket
0,O00001,delivered,1,1.0,10.82,delivered_late
1,O00002,delivered,1,1.0,3.97,delivered_late


In [6]:
grain_checks = pd.DataFrame([
    ("orders", "order_id", len(orders), orders.order_id.nunique()),
    ("customer_app_actions", "action_id", len(customer_actions), customer_actions.action_id.nunique()),
    ("customer_interactions", "interaction_id", len(interactions), interactions.interaction_id.nunique()),
    ("support_tickets", "ticket_id", len(tickets), tickets.ticket_id.nunique()),
    ("order_interventions", "intervention_id", len(interventions), interventions.intervention_id.nunique()),
    ("order_outcomes", "order_id", len(outcomes), outcomes.order_id.nunique()),
    ("order_events", "event_id", len(events), events.event_id.nunique()),
], columns=["source table", "claimed key", "rows", "distinct keys"])
grain_checks["key is unique?"] = grain_checks["rows"] == grain_checks["distinct keys"]
display(grain_checks)
display(interactions[interactions.interaction_id.duplicated(keep=False)].sort_values("interaction_id"))
print("interventions per order (max):", interventions.order_id.value_counts().max())
print("intervention types: interventions table", sorted(interventions.intervention_type.unique()),
      "| order_events", sorted(events.loc[events.source_system == "interventions", "event_type"].unique()))

,source table,claimed key,rows,distinct keys,key is unique?
0,orders,order_id,1603,1600,False
1,customer_app_actions,action_id,2365,2365,True
2,customer_interactions,interaction_id,497,494,False
3,support_tickets,ticket_id,202,201,False
4,order_interventions,intervention_id,430,430,True
5,order_outcomes,order_id,1600,1600,True
6,order_events,event_id,4955,4955,True


,interaction_id,order_id,interaction_type,interaction_at,channel,detail
174,CI-0178,O01453,SUPPORT_TICKET,2026-08-15T21:04:00,support,driver_not_moving
177,CI-0178,O01528,ORDER_STATUS_REFRESH,2026-08-08T17:44:00,customer_app,NaN
175,CI-0179,O01454,SUPPORT_TICKET,2026-08-09T13:21:00,support,driver_not_moving
178,CI-0179,O00618,ETA_VIEW,2026-08-26T20:24:00,customer_app,NaN
176,CI-0180,O01455,SUPPORT_TICKET,2026-08-16T19:11:00,support,ready_but_waiting
179,CI-0180,O00949,HELP_OPENED,2026-08-17T12:12:00,customer_app,NaN


interventions per order (max): 1
intervention types: interventions table ['CUSTOMER_CREDIT', 'DRIVER_REASSIGNMENT', 'PRIORITY_DISPATCH', 'RESTAURANT_CONTACT'] | order_events ['DRIVER_REASSIGNMENT', 'ETA_MESSAGE', 'PRIORITY_DISPATCH', 'REFUND_OFFER', 'RESTAURANT_CONTACT']


### The canonical project model (6 tables)

```
customer 1──* order 1──* customer_interaction   (app actions + support tickets, one row per touch)
                 │  1──* intervention            (what the business did)
                 │  1──1 order_outcome           (delivered? late? delay_min)
                 └──*── order_event              (lifecycle: created, assigned, picked_up, delivered)
```

| Table | Primary key | Important foreign keys | Grain |
|---|---|---|---|
| `customer` | `customer_id` | none | one customer |
| `order` | `order_id` (de-duplicated: the source has 3 duplicate IDs) | `customer_id`, `restaurant_id`, `driver_id` | one order |
| `order_event` | `event_id` | `order_id` | one lifecycle event (observed only) |
| `customer_interaction` | **new surrogate key** `(source, source_id)` | `order_id`, `customer_id` | one customer touch (app action, support ticket) |
| `intervention` | `intervention_id` | `order_id` | one business action on an order (max 1 per order today) |
| `order_outcome` | `order_id` | `order_id` → order (1:1) | one outcome per order |

**Why this beats mirroring every source table:**
- The project question follows **one order through its workflow**. The sources are organised by *system*: the app, support, dispatch, the ops console.
- The model reorganises them around **order → touches → actions → outcome**, so every KPI question becomes one join path instead of seven.
- **Source keys can't be trusted blindly:** `customer_interactions.interaction_id` is **reused** (CI-0178 to CI-0180 each name two different records), so the canonical table needs its own key.
- **Two sources disagree on interventions.** `order_interventions` (430 rows) has `CUSTOMER_CREDIT`; the event log (260 intervention events) has `ETA_MESSAGE`/`REFUND_OFFER`. The model names **one source of truth**: `order_interventions`, the system where the action is decided. The event log becomes a timing cross-check.

# Challenge 3 — Build interaction → intervention → outcome

Create one order-level table containing:

`order_id, customer_id, support_opened, cancel_attempted, intervention_count, intervention_types, final_status, late_flag, delay_min`

Answer:
1. How many late orders had support interaction?
2. How many orders received intervention?
3. Which intervention is most common?
4. Which frustrated journeys had no intervention?

### Hint
Aggregate one-to-many tables before joining them to order-level outcomes.

In [7]:
actions_by_order=(customer_actions.groupby("order_id").agg(action_count=("action_id","count")).reset_index())
# TODO: add flags, aggregate interventions, and join to outcomes/orders

In [8]:
flags = (customer_actions.pivot_table(index="order_id", columns="action_type", values="action_id", aggfunc="count", fill_value=0)
         .rename(columns=str.lower).reset_index())
ticket_orders = set(tickets.dropna(subset=["order_id"]).order_id)
iv = (interventions.groupby("order_id")
      .agg(intervention_count=("intervention_id", "count"),
           intervention_types=("intervention_type", lambda s: "|".join(sorted(set(s))))).reset_index())

base = orders.drop_duplicates("order_id")[["order_id", "customer_id", "restaurant_id", "driver_id"]]
order_level = (outcomes.merge(base, on="order_id", how="left")          # 1:1 on order_id
               .merge(flags, on="order_id", how="left")                 # already aggregated to one row per order
               .merge(iv, on="order_id", how="left")                    # already aggregated to one row per order
               .merge(actions_by_order, on="order_id", how="left"))
order_level[["eta_viewed", "support_opened", "cancel_attempted", "intervention_count", "action_count"]] = \
    order_level[["eta_viewed", "support_opened", "cancel_attempted", "intervention_count", "action_count"]].fillna(0).astype(int)
order_level["support_opened"] = (order_level["support_opened"] > 0) | order_level["order_id"].isin(ticket_orders)
order_level["cancel_attempted"] = order_level["cancel_attempted"] > 0
order_level["intervention_types"] = order_level["intervention_types"].fillna("")
order_level = order_level.rename(columns={"final_status_norm": "final_status"})
assert order_level["order_id"].is_unique and len(order_level) == len(outcomes), "join fan-out!"

cols = ["order_id", "customer_id", "support_opened", "cancel_attempted", "intervention_count", "intervention_types",
        "final_status", "late_flag", "delay_min"]
display(order_level[cols].head(8))
print("rows:", len(order_level), "| unique orders:", order_level.order_id.nunique())

,order_id,customer_id,support_opened,cancel_attempted,intervention_count,intervention_types,final_status,late_flag,delay_min
0,O00001,C0168,True,False,1,DRIVER_REASSIGNMENT,delivered,1.0,10.82
1,O00002,C0043,False,False,0,,delivered,1.0,3.97
2,O00003,C0855,False,False,0,,delivered,0.0,-4.41
3,O00004,C0229,False,False,0,,cancelled,NaN,NaN
4,O00005,C0040,False,False,0,,delivered,1.0,21.02
5,O00006,C0152,False,False,0,,delivered,1.0,7.30
6,O00007,C0875,False,False,0,,delivered,1.0,4.27
7,O00008,C0223,True,False,1,CUSTOMER_CREDIT,delivered,1.0,26.86


rows: 1600 | unique orders: 1600


In [9]:
late = order_level[order_level.late_flag == 1]
frustrated = order_level[(order_level.support_opened | order_level.cancel_attempted) & (order_level.intervention_count == 0)]
answers = pd.Series({
    "1. late orders with a support interaction": f"{int(late.support_opened.sum())} of {len(late)} ({100 * late.support_opened.mean():.1f}%)",
    "2. orders that received an intervention": int((order_level.intervention_count > 0).sum()),
    "3. most common intervention": interventions.intervention_type.value_counts().idxmax()
                                   + f" ({interventions.intervention_type.value_counts().max()})",
    "4. frustrated journeys (support or cancel attempt) with NO intervention": len(frustrated),
    "   ...of which delivered late": int((frustrated.outcome_bucket == "delivered_late").sum()),
})
display(answers.to_frame("answer"))
display(frustrated[cols].sort_values("delay_min", ascending=False).head(8))

,answer
1. late orders with a support interaction,258 of 843 (30.6%)
2. orders that received an intervention,430
3. most common intervention,DRIVER_REASSIGNMENT (155)
4. frustrated journeys (support or cancel attempt) with NO intervention,219
...of which delivered late,192


,order_id,customer_id,support_opened,cancel_attempted,intervention_count,intervention_types,final_status,late_flag,delay_min
193,O00194,C0771,True,False,0,,delivered,1.0,38.34
736,O00737,C0824,True,False,0,,delivered,1.0,36.28
302,O00303,C0411,True,False,0,,delivered,1.0,33.16
210,O00211,C0629,True,False,0,,delivered,1.0,32.38
1234,O01235,C0772,True,False,0,,delivered,1.0,32.00
976,O00977,C0254,True,False,0,,delivered,1.0,31.92
994,O00995,C0746,True,False,0,,delivered,1.0,31.72
1196,O01197,C0032,True,False,0,,delivered,1.0,31.61


**Answers:**
1. **258 of 843 late orders (30.6%)** had a support interaction (app "support opened" or a ticket).
2. **430 orders** received an intervention, exactly one each.
3. **`DRIVER_REASSIGNMENT` (155)** is the most common, then restaurant contact (116), priority dispatch (95) and customer credit (64).
4. **219 frustrated journeys got no intervention at all**, and **192** of them were delivered late. That is the clearest gap in the workflow: the customer signalled trouble and nothing happened.

*Aggregation discipline:* app actions and interventions were aggregated to one row per order *before* joining, and the final assert proves the table did not fan out.

# Challenge 4 — Select 3–5 business metrics

Project KPI: **Reduce Late Delivery Rate**.

For each chosen metric, document:
- metric name,
- formula,
- grain,
- why it matters,
- relationship to the project KPI.

At least one metric must represent:
- an outcome,
- a customer interaction,
- an intervention.

In [10]:
# Example starting point:
# valid_outcomes=outcomes[outcomes.late_flag.notna()]
# late_delivery_rate=valid_outcomes.late_flag.mean()

# TODO: calculate your selected metrics

In [11]:
o_t = orders.drop_duplicates("order_id").copy()
for col in ["created_at", "pickup_at", "actual_delivery_at"]:
    o_t[col] = pd.to_datetime(o_t[col], format="mixed")
o_t["order_to_pickup_min"] = (o_t["pickup_at"] - o_t["created_at"]).dt.total_seconds() / 60
ol = order_level.merge(o_t[["order_id", "order_to_pickup_min", "pickup_at", "actual_delivery_at"]], on="order_id")
valid = ol[ol.late_flag.notna()]

it = interventions.merge(ol[["order_id", "pickup_at", "actual_delivery_at", "late_flag"]], on="order_id")
it["iv_time"] = pd.to_datetime(it["intervention_at"], format="mixed")
it["phase"] = np.select([it.iv_time < it.pickup_at, it.iv_time <= it.actual_delivery_at, it.actual_delivery_at.isna()],
                        ["before pickup", "in transit", "no delivery time"], "after delivery")

metrics = pd.DataFrame([
    ("M1 Late delivery rate (OUTCOME)", "late / delivered orders with a known delivery time", "order",
     f"{100 * valid.late_flag.mean():.1f}% ({int(valid.late_flag.sum())}/{len(valid)})", "IS the project KPI"),
    ("M2 Order-to-pickup time, late vs on-time (WORKFLOW DRIVER)", "median(pickup_at - created_at) by late_flag", "order",
     f"{valid[valid.late_flag == 1].order_to_pickup_min.median():.1f} vs {valid[valid.late_flag == 0].order_to_pickup_min.median():.1f} min",
     "the stage where delay accumulates; the leading indicator"),
    ("M3 Support contact rate on late orders (INTERACTION)", "late orders with support opened or a ticket / late orders", "order",
     f"{100 * valid[valid.late_flag == 1].support_opened.mean():.1f}%", "the customer cost of each late order"),
    ("M4 Preventive intervention share (INTERVENTION)", "interventions made before pickup / all interventions", "intervention",
     f"{100 * (it.phase == 'before pickup').mean():.1f}% ({int((it.phase == 'before pickup').sum())}/{len(it)})",
     "only pre-pickup actions can change the KPI; the rest are recovery"),
    ("M5 Unattended frustration rate (INTERACTION x INTERVENTION)", "frustrated journeys with no intervention / frustrated journeys", "order",
     f"{100 * len(frustrated) / (ol.support_opened | ol.cancel_attempted).sum():.1f}% ({len(frustrated)}/{int((ol.support_opened | ol.cancel_attempted).sum())})",
     "missed chances to intervene"),
], columns=["metric", "formula", "grain", "value", "link to KPI"])
display(metrics)
display(pd.crosstab(it.intervention_type, it.phase, margins=True))

,metric,formula,grain,value,link to KPI
0,M1 Late delivery rate (OUTCOME),late / delivered orders with a known delivery time,order,56.4% (843/1495),IS the project KPI
1,"M2 Order-to-pickup time, late vs on-time (WORKFLOW DRIVER)",median(pickup_at - created_at) by late_flag,order,31.9 vs 19.8 min,the stage where delay accumulates; the leading indicator
2,M3 Support contact rate on late orders (INTERACTION),late orders with support opened or a ticket / late orders,order,30.6%,the customer cost of each late order
3,M4 Preventive intervention share (INTERVENTION),interventions made before pickup / all interventions,intervention,58.8% (253/430),only pre-pickup actions can change the KPI; the rest are recovery
4,M5 Unattended frustration rate (INTERACTION x INTERVENTION),frustrated journeys with no intervention / frustrated journeys,order,74.5% (219/294),missed chances to intervene


phase,after delivery,before pickup,in transit,no delivery time,All
intervention_type,,,,,
CUSTOMER_CREDIT,45,1,17,1,64
DRIVER_REASSIGNMENT,0,117,37,1,155
PRIORITY_DISPATCH,0,64,29,2,95
RESTAURANT_CONTACT,0,71,43,2,116
All,45,253,126,6,430


| Metric | Type | Formula | Grain | Value | Why it matters / link to the KPI |
|---|---|---|---|---|---|
| **M1 Late delivery rate** | outcome | late / delivered with known time | order | **56.4%** (843 / 1,495) | It *is* the project KPI (definition caveats from Class 6 apply). |
| **M2 Order-to-pickup time** | workflow driver | median(pickup − created), late vs on time | order | **31.9 vs 19.8 min** | The stage where delay accumulates; ops can act on it *before* the order is late. |
| **M3 Support contact rate on late orders** | customer interaction | late orders with support / late orders | order | **30.6%** | What each late order costs in support load and trust. |
| **M4 Preventive intervention share** | intervention | interventions before pickup / all | intervention | **58.8%** (253 / 430) | Only pre-pickup actions can move M1. All 45 post-delivery credits are recovery, not prevention. |
| **M5 Unattended frustration rate** | interaction × intervention | frustrated journeys with no intervention / frustrated | order | **74.5%** (219 / 294) | Directly controllable: the workflow gap between signal and action. |

# Challenge 5 — Investigate the workflow with joins and aggregations

Answer at least three:

A. Do orders with support interactions have higher delay?  
B. What is late rate with vs without intervention?  
C. Which intervention type is associated with the lowest late rate?  
D. Which restaurants contribute the largest number of late orders?  
E. Which journeys show support interaction + intervention + still late?

For every answer, add one sentence:

> **What does this tell the business, and what does it NOT prove?**

In [12]:
# TODO: use the order-level model plus joins/groupby
# Reminder: association != causation

In [13]:
print("A. Delay by support interaction")
display(valid.groupby("support_opened").agg(orders=("late_flag", "size"), late_rate=("late_flag", "mean"),
                                            median_delay=("delay_min", "median")).round(3))

print("B. Late rate with vs without intervention (all timings)")
display(valid.assign(intervened=valid.intervention_count > 0).groupby("intervened")
        .agg(orders=("late_flag", "size"), late_rate=("late_flag", "mean")).round(3))

print("C. Late rate by intervention type — all timings, then preventive (before pickup) only")
display(it[it.late_flag.notna()].groupby("intervention_type").late_flag.agg(["size", "mean"]).round(3))
display(it[(it.late_flag.notna()) & (it.phase == "before pickup")].groupby("intervention_type").late_flag.agg(["size", "mean"]).round(3))

print("D. Restaurants contributing the most late orders")
top = valid.groupby("restaurant_id").late_flag.agg(late_orders="sum", orders="size", late_rate="mean")
display(top.sort_values("late_orders", ascending=False).head(6).round(3))
print("share of all late orders from the top 10 restaurants:",
      f"{100 * top.late_orders.nlargest(10).sum() / top.late_orders.sum():.1f}%  (10 of {len(top)} restaurants = {100 * 10 / len(top):.0f}%)")

print("E. Journeys with support + intervention + still late")
e = ol[ol.support_opened & (ol.intervention_count > 0) & (ol.late_flag == 1)]
print(len(e), "orders; intervention types:", e.intervention_types.value_counts().to_dict())

A. Delay by support interaction


,orders,late_rate,median_delay
support_opened,,,
False,1203,0.486,-0.250
True,292,0.884,12.285


B. Late rate with vs without intervention (all timings)


,orders,late_rate
intervened,,
False,1091,0.564
True,404,0.564


C. Late rate by intervention type — all timings, then preventive (before pickup) only


,size,mean
intervention_type,,
CUSTOMER_CREDIT,63,0.667
DRIVER_REASSIGNMENT,147,0.578
PRIORITY_DISPATCH,86,0.512
RESTAURANT_CONTACT,108,0.528


,size,mean
intervention_type,,
CUSTOMER_CREDIT,1,1.000
DRIVER_REASSIGNMENT,110,0.736
PRIORITY_DISPATCH,57,0.702
RESTAURANT_CONTACT,65,0.692


D. Restaurants contributing the most late orders


,late_orders,orders,late_rate
restaurant_id,,,
R050,22.0,35,0.629
R051,20.0,29,0.690
R004,20.0,30,0.667
R030,20.0,38,0.526
R024,20.0,25,0.800
R023,20.0,32,0.625


share of all late orders from the top 10 restaurants: 23.3%  (10 of 60 restaurants = 17%)
E. Journeys with support + intervention + still late
67 orders; intervention types: {'DRIVER_REASSIGNMENT': 26, 'CUSTOMER_CREDIT': 15, 'PRIORITY_DISPATCH': 14, 'RESTAURANT_CONTACT': 12}


| Question | Finding | What it tells the business | What it does **not** prove |
|---|---|---|---|
| **A** Support vs delay | Orders with support contact: **88.4% late**, median +12.3 min, vs 48.6% late / −0.3 min without | Support contact is a strong real-time *signal* of a failing order | That contacting support makes orders late: the causation runs the other way |
| **B** With vs without intervention | **56.4% vs 56.4%**, identical | Interventions do not visibly *lower* the late rate as currently used | That interventions are useless: they are aimed at orders already at risk (selection bias) |
| **C** By type | Priority dispatch has the lowest late rate overall (51.2%); **pre-pickup** interventions all sit at **69–74% late** | Pre-pickup interventions go to the riskiest orders (above the 56% base) | Which intervention *works*. That needs a comparison against similar-risk orders without intervention, or an A/B test |
| **D** Restaurants | R050 has 22 late orders; R051, R004, R030, R024 and R023 have 20 each | Late orders are spread widely: the top 10 restaurants (17% of them) hold only 23.3% of late orders, so this is a system problem, not a few bad restaurants | That those restaurants are slow. Pickup delay mixes food readiness and driver arrival (no arrival event) |
| **E** Support + intervention + still late | **67 orders** | Intervened too late, or with the wrong lever; a list worth reviewing case by case | Why each one failed, without the per-order timeline review |

# Challenge 6 — Connect the model to the KPI

Create a one-page project view:

`PROJECT KPI → OUTCOME METRIC → WORKFLOW/DRIVER METRICS → INTERVENTIONS → DATA SOURCES/EVENTS`

Answer:
1. Which metrics are directly controllable by operations?
2. Which are outcomes?
3. Which missing event limits the model most?
4. What would you instrument next?

Final deliverable:
- core entities,
- key events,
- relationships,
- 3–5 metrics,
- KPI linkage,
- one modelling limitation.

### One-page project view

```
PROJECT KPI         Reduce late delivery rate
   │
OUTCOME METRIC      M1 Late delivery rate: 56.4% of delivered orders (843/1,495)
   │
WORKFLOW / DRIVERS  M2 order-to-pickup time (31.9 vs 19.8 min) ◄── where delay accumulates
                    dispatch pickup slip vs plan · ETA revisions · reassignment (95 orders)
   │
INTERACTIONS        M3 support contact on late orders (30.6%) · ETA views · cancel attempts
   │
INTERVENTIONS       M4 preventive share (58.8% before pickup) · M5 unattended frustration (74.5%)
                    driver reassignment · restaurant contact · priority dispatch · customer credit
   │
SOURCES / EVENTS    orders + order_events (lifecycle) · dispatch API (assignment, ETA) · customer_app_actions
                    + support_tickets (interactions) · order_interventions (actions) · order_outcomes
```

1. **Directly controllable by operations:**
   - **M4** (intervene before pickup, not after delivery).
   - **M5** (respond to every frustration signal).
   - The *inputs* to **M2**: dispatch timing, restaurant contact, reassignment speed.
2. **Outcomes:** **M1** (late rate) and **M3** (support load) are results, not levers.
3. **The missing event that limits the model most:** **`driver_arrived_at_restaurant`**, together with **`food_ready`**.
   - Without them, M2 (the biggest delay component) cannot be split into *restaurant not ready* vs *driver not there*.
   - So the model cannot say which intervention (restaurant contact vs reassignment) fits which late order.
4. **Instrument next:**
   - A geofenced driver-arrival event and a restaurant "food ready" tap, both as immutable events with IDs.
   - A timestamped **intervention decision log** that records the risk score at decision time, so effectiveness can be measured against comparable orders.
   - Unique interaction IDs across the app and support.

**Final deliverable summary:**

| Element | Our model |
|---|---|
| Core entities | customer, order, customer_interaction, intervention, order_outcome (+ order_event) |
| Key events | created, assigned (/ reassigned), picked up, delivered; app actions, tickets, interventions |
| Relationships | customer 1:N order; order 1:N events / interactions / interventions; order 1:1 outcome |
| Metrics | M1 late rate · M2 order-to-pickup · M3 support on late · M4 preventive share · M5 unattended frustration |
| KPI linkage | M2, M4 and M5 are levers; M1 is the KPI; M3 is its customer cost |
| **Modelling limitation** | intervention effectiveness is not identifiable: interventions target at-risk orders, and there is no arrival/ready event to attribute the pickup stage |

# Final reflection

A strong solution does not produce the largest schema.

It produces the **smallest useful model that explains the workflow and supports the project KPI**.